# 💊 Farmácias Boa Dose: Previsão de Demanda

Estudo de caso de **previsão de vendas (série temporal)** para uma rede fictícia de farmácias. O objetivo é ajudar o time de compras a decidir **quanto pedir de cada produto** nas próximas semanas, com menos falta de produto na prateleira e menos estoque parado.

**Sumário**

1. Entendimento do Negócio e Contextualização
2. Bibliotecas utilizadas
3. Carregamento dos dados
4. Descrição dos dados
5. Limpeza
6. Análise Exploratória
7. Modelagem
8. Conclusões e Recomendações
9. Deploy: como reproduzir

## 1. Entendimento do Negócio e Contextualização

### A empresa

A **Farmácias Boa Dose** é uma rede regional fundada em 2015 no Paraná. Hoje tem **5 lojas**: duas em Curitiba, uma em Londrina, uma em Maringá e uma em Ponta Grossa. O foco é o atendimento de bairro: medicamentos sem receita, higiene, beleza e cuidados com a saúde.

As compras são centralizadas. Um time pequeno faz o pedido para os fornecedores toda semana, e os produtos chegam no centro de distribuição de Curitiba antes de seguir para as lojas.

### O problema

Hoje o pedido é feito de um jeito bem simples: **"pede o mesmo tanto que vendeu na semana passada"**. Na maior parte do ano isso funciona, mas quebra justo nos momentos que mais importam:

- **No inverno**, a procura por antigripais e xaropes sobe rápido. Como o pedido segue a semana anterior, ele fica sempre atrasado, e as lojas ficam **sem produto na prateleira (ruptura)** no pico da gripe.
- **No fim do verão**, acontece o contrário: o protetor solar continua sendo pedido no ritmo de janeiro e sobra **estoque parado**, ocupando espaço e capital.
- **Promoções e feriados** mudam as vendas de um jeito que a regra atual não enxerga.

Em 2025 a diretoria percebeu que a regra estava custando caro dos dois lados: vendas perdidas por falta de produto e dinheiro preso em estoque. Por isso pediu um estudo usando o histórico de vendas.

### O objetivo

Entender como as vendas se comportam ao longo do tempo (tendência, sazonalidade, promoções e feriados) e construir uma **previsão semanal de vendas por produto** para as próximas **4 semanas** (janeiro de 2026). A previsão precisa errar **menos** que a regra atual.

### Os dados

Histórico de vendas diárias de **2023-01-01 a 2025-12-31** (3 anos), com cerca de **20 produtos** nas 5 lojas. Os dados vêm do sistema das lojas e têm problemas comuns de base real, que serão tratados na limpeza.

### Perguntas de negócio

1. Como as vendas da rede evoluíram de 2023 a 2025? Existe tendência de crescimento?
2. Quais produtos têm sazonalidade forte e em quais meses ficam no pico?
3. Quanto as promoções e os feriados mexem nas vendas?
4. As lojas se comportam de forma parecida ou alguma foge do padrão?
5. Dá para prever as vendas semanais das próximas 4 semanas errando menos que a regra atual ("repete a semana passada")?
6. Quanto pedir de cada produto nas próximas 4 semanas?

### Estrutura do projeto

Todos os arquivos ficam na raiz da pasta:

```
pharmacy-demand-forecast/
├── pharmacy_demand_forecast.ipynb   # notebook com a análise completa
├── generate_data.py                 # script que gera as bases fictícias
├── stores.csv                       # cadastro das lojas
├── products.csv                     # cadastro dos produtos
├── sales.csv                        # vendas diárias por loja e produto
├── holidays.csv                     # feriados do período
├── requirements.txt                 # bibliotecas do ambiente
├── README.md                        # resumo do projeto
└── .gitignore
```

## 2. Bibliotecas utilizadas

As bibliotecas entram aqui conforme forem usadas ao longo do projeto.

In [1]:
import pandas as pd

## 3. Carregamento dos dados

Os dados estão em 4 arquivos CSV, gerados pelo script `generate_data.py`:

| Arquivo | Conteúdo |
|---|---|
| `stores.csv` | Cadastro das 5 lojas |
| `products.csv` | Cadastro dos 20 produtos, com categoria e preço |
| `holidays.csv` | Feriados nacionais do período |
| `sales.csv` | Vendas diárias por loja e produto, de 2023 a 2025 |

In [3]:
stores = pd.read_csv('stores.csv')
products = pd.read_csv('products.csv')
holidays = pd.read_csv('holidays.csv')
sales = pd.read_csv('sales.csv')

print('stores:', stores.shape)
print('products:', products.shape)
print('holidays:', holidays.shape)
print('sales:', sales.shape)

stores: (5, 4)
products: (20, 4)
holidays: (39, 2)
sales: (108543, 6)


In [4]:
display(stores)
print('\n')
display(products.head())
print('\n')
display(holidays.head())
print('\n')
display(sales.head())

,store_id,store_name,city,opening_date
0,S01,Curitiba Centro,Curitiba,2015-03-10
1,S02,Curitiba Batel,Curitiba,2017-08-21
2,S03,Londrina,Londrina,2018-05-14
3,S04,Maringá,Maringá,2019-11-04
4,S05,Ponta Grossa,Ponta Grossa,2021-02-08


,product_id,product_name,category,unit_price
0,P01,Antigripal Comprimidos,Gripe e resfriado,18.9
1,P02,Xarope para Tosse,Gripe e resfriado,24.5
2,P03,Descongestionante Nasal,Gripe e resfriado,15.9
3,P04,Pastilha para Garganta,Gripe e resfriado,12.9
4,P05,Vitamina C Efervescente,Vitaminas,21.9


,date,holiday_name
0,2023-01-01,Confraternização Universal
1,2023-02-20,Carnaval
2,2023-02-21,Carnaval
3,2023-04-07,Sexta-feira Santa
4,2023-04-21,Tiradentes


,date,store_id,product_name,units_sold,unit_price,on_promotion
0,2023-01-01,S01,Antigripal Comprimidos,3,18.9,0
1,2023-01-01,S01,Xarope para Tosse,2,24.5,0
2,2023-01-01,S01,Descongestionante Nasal,3,15.9,0
3,2023-01-01,S01,Pastilha para Garganta,3,12.9,0
4,2023-01-01,S01,VITAMINA C EFERVESCENTE,2,21.9,0
